# Building AI News Developer Agent with Google ADK

## Getting Started with ADK  

This `Notebbook_app_001.ipynb` demonstrates:

- Setting up a new agent folder (`adk create`)
- Writing the first `agent.py`
- Adding text models and built-in tools (`google_search`, `BuiltInCodeExecutor`)
- Fine-tuning agent instructions and behavior
- Testing valid and invalid prompts

In [1]:
# Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv()
import os 

In [2]:
print("HF configured:", bool(os.getenv("HUGGING_FACE_TOKEN")))
print("GitHub configured:", bool(os.getenv("GITHUB_TOKEN")))


HF configured: True
GitHub configured: True


## Setting up the agent  

Run the cell below to create the folder structure for the agent.  

In [ ]:
!adk create --type=code app_01 --model gemini-2.5-flash --api_key $GEMINI_API_KEY  

- Using the `adk create` command, the below new folder structure with ADK's built-in project scaffolding is set up,  generating three essential files:  

File structure:  

```bash 
app_01/
    __init__.py   # The `__init__.py` file marks the directory as a Python package, nabling proper imports.   
    agent.py      # he `agent.py` file provides a clean foundation where you'll implement your agent.  
    .env          # The `.env` file securely stores your API credentials and configuration.
```

>_Note_ : In this the project `--type=code` option has been selected to generate a Python-based agent in `agent.py`.

- Adding a text model    

The `--model` parameter specifies the LLM to be used by the agent. Here it will be used a text-focused model like `gemini-2.5-flash` since this is ideal when the purpose is to optimize text processing and to provide faster response times.  

-  Adding an api key  /Set up Google API key and Vertex AI based authentication    
  
The `--api_key` parameter specifies the api key to be used by the agent. You can create a [Gemini API key](https://docs.cloud.google.com/vertex-ai/generative-ai/docs/start/quickstart?usertype=apikey#python-gen-ai-sdk). This key is essential for authenticating your requests to the Gemini API. A step-by-step guide can be found on o [Google AI Studio](https://cloud.google.com/free?hl=en)

## Writing the `agent.py`

`adk create` command is used to create folders and then write to its `agent.py` using the specific command, `%%writefile FILENAME`, to interact with the files in the new agent folder. 

### Adding tools to the agent  

**But here's the problem:** try asking this agent about the latest AI developments, and you'll quickly discover it can only tell you about things that happened before its training cutoff date. For an AI news assistant that's supposed to fetch the latest news, that's not particularly helpful.

Therefore, you need to fix that by providing your agent with **Tools**. In Google ADK, the word [“tool”](https://google.github.io/adk-docs/tools/) has two meanings:  

| Term                 | Meaning                                                                   |
| -------------------- | ------------------------------------------------------------------------- |
| **ADK Tool**         | A callable object exposed to an agent (e.g. `google_search`, `AgentTool`) |
| **Third-Party Tool** | Any external system or API                                                |

ADK comes with several `powerful built-in tools`, such as `Gemini tools` (e.g.: `google_search` and `Code Execution`) and `Third-party tools` (e.g.: `HuggingFace`, `Github`), but these ADK Tools have some `Restrictions`. Thus, they  typically cannot be combined within a single agent instance. Therefore, it has been followed the below workaround: 

Workaround:  
- Create specialized agents (e.g. SearchAgent, CodeAgent)
- Orchestrate them exclusively via the RootAgent
- Delegate requests using AgentTool.create()
This preserves architectural clarity while respecting ADK constraints.

Therefore, the below architectural workflow had been followed: 
```bash  
RootAgent
   ├── AgentTool → AIDevSearchAgent → google_search
   ├── AgentTool → CodeAgent → BuiltInCodeExecutor
   ├── AgentTool → CodeExplainAgent    
   ├── AgentTool → hugging_face_agent → HF MCP → HF Hub
   └── AgentTool → github_agent → GitHub MCP → GitHub
``` 
> _Note_: Final architectural rule :   
          - APIs belong to agents.    
          - Agents belong to RootAgent.  
>         - RootAgent never talks to APIs directly.  

The application is built using a **state-aware, multi-agent architecture powered by Gemini models and Google ADK Web**, designed specifically for AI developers.   

### Fine-tuning agent instructions

So far the agent has simple instructions, but for reliable behavior, more sophisticated instruction engineering have been enclosed. Therefore, the agent has been enhanced  with strict behavioral controls.  

In [ ]:
%%writefile app_01/agent.py
import os
import re
from google.adk.agents import Agent
from google.adk.tools import google_search
from google.adk.tools.agent_tool import AgentTool
from google.adk.code_executors import BuiltInCodeExecutor

# ======================================================
# Environment variables
# ======================================================
HF_TOKEN = os.getenv("HUGGING_FACE_TOKEN")
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")
# ======================================================
# Helper functions
# ======================================================
def extract_headlines(text: str):
    return re.findall(r"\d+\.\s*(.+)", text)


def extract_repo_candidate(text: str):
    """
    STRICT GitHub repo extraction.
    Returns owner/repo or None.
    """
    match = re.search(r"(?:github\.com/)?([\w\-]+/[\w\-]+)", text)
    return match.group(1) if match else None


def extract_hf_candidate(text: str):
    """
    STRICT Hugging Face ID extraction: org/name
    """
    match = re.search(r"\b([\w\-]+/[\w\-]+)\b", text)
    return match.group(1) if match else None


# ======================================================
# AI Developer News Agent
# ======================================================
search_agent = Agent(
    model="gemini-2.5-flash",
    name="AIDevSearchAgent",
    description="AI developer news analyst with structured output.",
    instruction="""
You are an AI News Analyst for developers.

Rules:
- ONLY AI-related developer news
  If asked anything else, respond: "I can only provide recent AI-related developer news."
- ALWAYS use google_search
- DEFAULT to 3 articles if no number specified
- NEVER ask follow-up questions

Required output:

Using google_search, here are the top headlines:

---
[NUMBER]. HEADLINE

Summary:
1–2 sentence technical summary

Tech stack:
- frameworks / languages / infra OR Not mentioned

License:
- Open-source | Proprietary | Mixed | Not mentioned

GitHub repository:
- owner/repo if mentioned
- Otherwise: Not referenced

Hugging Face:
- Model / Dataset / Space if mentioned
- Otherwise: Not mentioned

Who should care:
- ML Engineer / Backend / MLOps / Data Scientist
---

End with:
"Which headline would you like to explore in more detail?"
""",
    tools=[google_search],
)

# ======================================================
# Python Execution Agent
# ======================================================
coding_agent = Agent(
    model="gemini-2.5-flash",
    name="CodeAgent",
    description="Safe Python execution agent.",
    instruction="""
Execute Python safely.

Rules:
- No filesystem access
- No network calls
- No infinite loops
- Return result or error only
""",
    code_executor=BuiltInCodeExecutor(),
)

# ======================================================
# Python Explanation Agent
# ======================================================
code_explain_agent = Agent(
    model="gemini-2.5-flash",
    name="CodeExplainAgent",
    description="Explains Python code safely.",
    instruction="""
Explain Python code step by step.
Do NOT execute or modify code.
""",
)

# ======================================================
# Hugging Face Canonical Reference Agent
# ======================================================
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from mcp import StdioServerParameters

hf_agent = Agent(
    model="gemini-2.5-flash",
    name="hugging_face_agent",
    description="Returns canonical Hugging Face URLs for exact IDs.",
    instruction="""
Rules:
- ONLY accept exact Hugging Face IDs (org/name)
- NEVER guess or infer
- If not found, respond exactly:
"No Hugging Face resource found for the provided identifier."

Valid output ONLY:

Hugging Face URL:
https://huggingface.co/<exact_id>
""",
    tools=(
        [
            McpToolset(
                connection_params=StdioConnectionParams(
                    server_params=StdioServerParameters(
                        command="npx",
                        args=["-y", "@llmindset/hf-mcp-server"],
                        env={"HF_TOKEN": HF_TOKEN},
                    ),
                    timeout=30,
                ),
            )
        ]
        if HF_TOKEN
        else []
    ),
)

# ======================================================
# GitHub MCP Agent (STRICT, DATA-ONLY)
# ======================================================
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPServerParams

git_agent = Agent(
    model="gemini-2.5-flash",
    name="github_agent",
    description="GitHub MCP repository inspector.",
    instruction="""
Input will be EXACT owner/repo.

MANDATORY:
- Call GitHub MCP
- No guessing
- No prose
- If MCP fails, say exactly:
  "No GitHub repository found."

Output format ONLY:

Repository: owner/repo
Stars: <number>
Forks: <number>
Open issues: <number>
Open PRs: <number>
Recent activity:
- Commits (30d): <number>
- Last commit date: <date>
Overall activity level: High | Medium | Low
""",
    tools=[
        McpToolset(
            connection_params=StreamableHTTPServerParams(
                url="https://api.githubcopilot.com/mcp/",
                headers={
                    "Authorization": f"Bearer {GITHUB_TOKEN}",
                    "X-MCP-Toolsets": "all",
                    "X-MCP-Readonly": "true",
                },
            )
        )
    ] if GITHUB_TOKEN else [],
)

# ======================================================
# Root Routing Agent
# ======================================================
root_agent = Agent(
    name="RootAgent",
    model="gemini-2.5-flash",
    description="Strict routing agent.",
    instruction="""
Routing rules:
- AI news → AIDevSearchAgent
- Python execution → CodeAgent
- Python explanation → CodeExplainAgent
- Hugging Face links → hugging_face_agent
- GitHub repos → github_agent

Rules:
- ALWAYS delegate
- NEVER answer directly
""",
    tools=[
        AgentTool(agent=search_agent),
        AgentTool(agent=coding_agent),
        AgentTool(agent=code_explain_agent),
        AgentTool(agent=hf_agent),
        AgentTool(agent=git_agent),
    ],
)

# ======================================================
# Main Input Handler
# ======================================================
def handle_user_input(user_input: str, session_state: dict):
    if session_state is None:
        session_state = {"headlines": []}

    if user_input.lower() in {"exit", "quit"}:
        return "Goodbye!", {"headlines": []}

    # Python execution
    if user_input.lower().startswith("execute python code:"):
        code = user_input[len("execute python code:"):].strip()
        return coding_agent.run(code), session_state

    # Python explanation
    if user_input.lower().startswith("explain this python code:"):
        code = user_input[len("explain this python code:"):].strip()
        return code_explain_agent.run(code), session_state

# 🔴 FORCE GitHub routing FIRST
    repo = extract_repo_candidate(user_input)
    if repo and "/" in repo:
        return git_agent.run(repo), session_state

    # Python execution
    #if user_input.lower().startswith("execute python code:"):
        #code = user_input.split(":", 1)[1]
        #return coding_agent.run(code), session_state


# ️⃣ Explicit Hugging Face ID → MCP
    hf_id = extract_hf_candidate(user_input)
    if hf_id and "/" in hf_id:
        hf_result = hf_agent.run(hf_id)
        if hf_result:
            return hf_result, session_state

 # Default routing
    response = root_agent.run(user_input)
    session_state["headlines"] = extract_headlines(response)
    return response, session_state

